# SeqTrainer Tutorial 05: End-to-end mini CNN regressor

This notebook mirrors Tutorial 04, but uses a **regression head** to predict promoter activity (`target`) directly.

## Goal

Demonstrate an end-to-end regression workflow with a short training run.

Increase `NUM_CYCLES` later for longer training.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from seqtrainer.data.sbol import build_dataset_from_files
from seqtrainer.data.materialized import MaterializedDataset
from seqtrainer.transforms.dna import one_hot_encode, pad_or_trim

print("Imports loaded")

## 1) Load SBOL data for promoter activity regression

In [ ]:
base = Path("data/sbol_data")
files = sorted(base.glob("sample_design_*.xml"))[:40]
df = build_dataset_from_files(files)

if df.empty:
    raise RuntimeError("No rows were materialized from SBOL inputs.")

print(f"Rows: {len(df)}")
df[["sequence", "target"]].head()

## 2) DNA preprocessing (fixed length + one-hot)

In [ ]:
SEQ_LEN = 120

fixed_sequences = [pad_or_trim(seq, length=SEQ_LEN) for seq in df["sequence"].tolist()]
X = one_hot_encode(fixed_sequences)  # [N, L, C]
y = df["target"].to_numpy(dtype=np.float32)

# Conv1d input: [N, C, L]
X_t = torch.tensor(np.transpose(X, (0, 2, 1)), dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

X_t.shape, y_t.shape

## 3) Train/val/test split using `MaterializedDataset`

In [ ]:
examples = [{"idx": i} for i in range(len(y))]
materialized = MaterializedDataset(examples, metadata={"tutorial": "cnn_regression_demo"})
train_ds, val_ds, test_ds = materialized.train_val_test_split(0.7, 0.15, 0.15, seed=42)

def to_index_tensor(split):
    return torch.tensor([row["idx"] for row in split.examples], dtype=torch.long)

train_idx, val_idx, test_idx = map(to_index_tensor, (train_ds, val_ds, test_ds))
len(train_idx), len(val_idx), len(test_idx)

## 4) Build dataloaders

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(TensorDataset(X_t[train_idx], y_t[train_idx]), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_t[val_idx], y_t[val_idx]), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_t[test_idx], y_t[test_idx]), batch_size=BATCH_SIZE)

len(train_loader), len(val_loader), len(test_loader)

## 5) Define a compact CNN backbone + regression head

In [ ]:
class TinyDNACNNRegressor(nn.Module):
    def __init__(self, channels: int = 5):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = TinyDNACNNRegressor()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model

## 6) Train for 10 cycles (easy to increase)

In [ ]:
NUM_CYCLES = 10

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, total_mae, total = 0.0, 0.0, 0
    for xb, yb in loader:
        if train:
            optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_mae += (preds - yb).abs().sum().item()
        total += xb.size(0)
    return total_loss / total, total_mae / total

for cycle in range(1, NUM_CYCLES + 1):
    train_mse, train_mae = run_epoch(train_loader, train=True)
    val_mse, val_mae = run_epoch(val_loader, train=False)
    print(
        f"cycle={cycle:02d} "
        f"train_mse={train_mse:.4f} train_mae={train_mae:.4f} "
        f"val_mse={val_mse:.4f} val_mae={val_mae:.4f}"
    )

## 7) Quick test-set check

In [ ]:
test_mse, test_mae = run_epoch(test_loader, train=False)
print(f"test_mse={test_mse:.4f} test_mae={test_mae:.4f}")